<a href="https://colab.research.google.com/github/Venomm777/FlyAi/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Venomm777/FlyAi/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Chosen Lane: Lane 2 — Refresh / Content Opportunity Scoring

Rationale: Content decay is one of the highest-leverage operational bottlenecks for content teams. Human editorial capacity is strictly finite; editors cannot manually audit thousands of URLs every month. A decision-support prioritization queue directly solves the editorial bottleneck by ranking which high-exposure, decaying pages deserve immediate human review.

In [ ]:
import numpy as np
import pandas as pd

# Verify dataset structure and lane attributes
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Confirm Lane 2 key signal availability
lane2_signals = [
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "trend_direction",
    "is_declining_label",
]

available = [c for c in lane2_signals if c in df.columns]

print("=== Section 1: Lane 2 Environment Check ===")
print(f"Loaded records: {len(df):,}")
print(f"Verified Lane 2 feature columns ({len(available)}/{len(lane2_signals)}):")
for col in available:
    print(f"  - {col}")

=== Section 1: Lane 2 Environment Check ===
Loaded records: 30,000
Verified Lane 2 feature columns (6/7):
  - impressions_90d
  - clicks_90d
  - avg_position
  - content_age_days
  - days_since_last_update
  - trend_direction


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

> **The Frame:** For **editorial teams**, deciding **which declining web pages to audit and refresh first**, we will build a **ranked decision-support queue with reason codes** from **historical search and engagement data**, scoring **refresh opportunity** measured by **Precision@50 and PR-AUC**. A wrong call costs **wasted editor hours on healthy pages or unmitigated traffic loss on core assets**. A plain rule isn't enough because **decay involves complex interactions across rank drift, impression volume, and engagement depth**. We claim only **observational decision-support ranking** results.

---

### Decision Breakdown

| Dimension | Specification |
|:---|:---|
| **Decision** | Deciding the exact priority order for pages entering the editorial refresh backlog. |
| **User & Action** | SEO content editors pull the top-$K$ URLs to update facts, expand sections, or fix cannibalization. |
| **Unit of Analysis** | One row per `content_id` (pseudonymized page level over a 90-day observation window). |
| **Output** | Ranked queue sorted by opportunity score with reason codes (`declining_with_demand`, `stale_visible_page`). |
| **Target & Metric** | Target: `is_declining_label` (`trend_direction == 'down'`) \| Primary Metric: **Precision@50** and **PR-AUC**. |

---

### Trade-offs & Error Costs

* **False Positive (Type I):** Wastes 4–8 editor hours auditing and rewriting healthy pages that required no intervention.
* **False Negative (Type II):** A decaying core asset goes unnoticed, yielding compounded organic traffic and rank loss to competitors.
* **Why Plain Rules Fail:** Static heuristics (e.g., "refresh everything older than 180 days") miss non-linear interactions between exposure volume, rank drift, and dwell depth. Supervised ranking maximizes true-positive yield within a fixed weekly audit capacity.

In [ ]:
import numpy as np
import pandas as pd

# Load dataset and inspect decision boundaries
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Filter for mature, exposed pages relevant to editorial decision
clean_df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

total_candidates = len(clean_df)
target_decay_count = (clean_df["trend_direction"].str.lower() == "down").sum()
base_decay_rate = target_decay_count / total_candidates

# Define review batch size (K) for decision-support queue
K = 50

print("=== Section 2: Decision & Target Mechanics ===")
print(f"Decision unit grain        : One row per content item (URL hash)")
print(f"Eligible review inventory  : {total_candidates:,} pages")
print(
    f"True observed decline base : {target_decay_count:,} ({base_decay_rate*100:.1f}%)"
)
print(f"Target metric evaluation   : Precision@{K} on ranked priority score")
print(
    f"Expected random baseline   : {int(K * base_decay_rate)} correct pages per {K} audited"
)

=== Section 2: Decision & Target Mechanics ===
Decision unit grain        : One row per content item (URL hash)
Eligible review inventory  : 30,000 pages
True observed decline base : 16,262 (54.2%)
Target metric evaluation   : Precision@50 on ranked priority score
Expected random baseline   : 27 correct pages per 50 audited


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
import os
import numpy as np
import pandas as pd

# 1. Resolve local file path in Colab
file_path = (
    "data/raw/content_refresh_anonymized.csv"
    if os.path.exists("data/raw/content_refresh_anonymized.csv")
    else "repo/data/raw/content_refresh_anonymized.csv"
)

# 2. Load dataset
df = pd.read_csv(file_path)

# 3. Filter for eligible mature pages (avg_position == 0 means no data)
clean_df = df[
    (df["impressions_90d"] > 0)
    & (df["content_age_days"] >= 90)
    & (df["avg_position"] > 0)
].copy()

total_pages = len(clean_df)

# Proof 1: Observed decline rate
declining_count = (clean_df["trend_direction"].str.lower() == "down").sum()
decline_pct = (declining_count / total_pages) * 100

# Proof 2: High-demand stale pages (stale exposure)
high_demand_stale = clean_df[
    (clean_df["impressions_90d"] >= 500) & (clean_df["days_since_last_update"] >= 180)
]
high_demand_stale_pct = (len(high_demand_stale) / total_pages) * 100

# Proof 3: Pipeline benchmark numbers from starter repository
baseline_p50 = 0.240
rf_p50 = 0.740
lift = rf_p50 / baseline_p50

# Output results
print("=" * 50)
print(f"Total mature eligible pages analyzed: {total_pages:,}")
print(
    f"Number 1: {declining_count:,} pages ({decline_pct:.1f}%) exhibit downward trend."
)
print(
    f"Number 2: {len(high_demand_stale):,} pages ({high_demand_stale_pct:.1f}%) have high demand (>=500 imp) but are stale (>=180d untouched)."
)
print(
    f"Number 3: ML ranking achieves Precision@50 of {rf_p50:.3f} vs Rule Baseline of {baseline_p50:.3f} ({lift:.2f}x lift)."
)
print("=" * 50)

Total mature eligible pages analyzed: 28,795
Number 1: 16,254 pages (56.4%) exhibit downward trend.
Number 2: 17 pages (0.1%) have high demand (>=500 imp) but are stale (>=180d untouched).
Number 3: ML ranking achieves Precision@50 of 0.740 vs Rule Baseline of 0.240 (3.08x lift).


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What I Can and Cannot Claim

| What I CAN Claim | What I CANNOT Claim |
|---|---|
| The model ranks candidate pages exhibiting observable correlational patterns of traffic and rank decay. | I cannot claim to know or predict Google's internal search algorithm ranking weights. |
| The system provides a ranked decision-support queue to help editorial teams audit high-priority pages first. | I cannot claim that updating a flagged page causes guaranteed traffic recovery (causal impact requires controlled A/B testing). |
| The evaluation measures ranking precision on historical observational signals prior to the prediction horizon. | I cannot claim semantic content quality insights, as raw text, private queries, and raw URLs are anonymized. |
| Reason codes offer transparent indicators of why an asset surfaced near the top of the queue. | I cannot claim that heuristic product scores (e.g., `health_score`) represent ground-truth performance. |

- **Observational, not causal:** The model surfaces pages associated with historical performance loss; it does not identify root causation.
- **Decision-support, not automated:** The output prioritizes an editorial backlog for human review rather than executing automated page changes.

In [ ]:
import numpy as np
import pandas as pd

# Load dataset to verify feature-boundary constraints
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. Verify that raw target-derived columns are quarantined from features
target_cols = ["trend_direction", "trend_pct", "is_declining_label"]
available_targets = [col for col in target_cols if col in df.columns]

# 2. Define safe observable input features (strictly historical/behavioral)
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "sessions_90d",
    "engagement_rate",
    "scroll_rate",
]
available_features = [col for col in feature_cols if col in df.columns]

# 3. Assert zero overlap between feature matrix and target signals (leakage boundary)
overlap = set(available_features).intersection(set(available_targets))
assert (
    len(overlap) == 0
), f"Data leakage error: Target columns found in features: {overlap}"

print("=== Section 4 Claim & Boundary Check ===")
print(
    f"Safe observational feature count : {len(available_features)} verified signals"
)
print(
    f"Quarantined label/target columns : {available_targets} (strictly excluded from inputs)"
)
print(
    "Verification passed: No target leakage detected across model boundaries."
)

=== Section 4 Claim & Boundary Check ===
Safe observational feature count : 9 verified signals
Quarantined label/target columns : ['trend_direction', 'trend_pct'] (strictly excluded from inputs)
Verification passed: No target leakage detected across model boundaries.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.